In [ ]:
# Fast BSE Stock Analysis Tool - Optimized for Speed
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
warnings.filterwarnings('ignore')

# Optimize pandas for speed
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 100)

In [ ]:
class FastBSEAnalyzer:
    def __init__(self):
        # Use smaller, high-quality stock list for speed
        self.top_stocks = [
            'RELIANCE.BO', 'TCS.BO', 'HDFCBANK.BO', 'INFY.BO', 'HINDUNILVR.BO',
            'ICICIBANK.BO', 'KOTAKBANK.BO', 'SBIN.BO', 'BHARTIARTL.BO', 'ITC.BO',
            'ASIANPAINT.BO', 'LT.BO', 'AXISBANK.BO', 'MARUTI.BO', 'SUNPHARMA.BO',
            'TITAN.BO', 'ULTRACEMCO.BO', 'WIPRO.BO', 'NESTLEIND.BO', 'POWERGRID.BO',
            'NTPC.BO', 'BAJFINANCE.BO', 'HCLTECH.BO', 'ONGC.BO', 'TATAMOTORS.BO'
        ]
        self.data = {}
        
    def fetch_single_stock_fast(self, symbol):
        """Ultra-fast single stock data fetch"""
        try:
            ticker = yf.Ticker(symbol)
            # Get minimal data for speed
            hist = ticker.history(period='2d')
            
            if len(hist) >= 1:
                current = hist['Close'].iloc[-1]
                prev = hist['Close'].iloc[-2] if len(hist) > 1 else current
                change_pct = ((current - prev) / prev * 100) if prev != 0 else 0
                
                return {
                    'symbol': symbol.replace('.BO', ''),
                    'price': round(current, 2),
                    'change%': round(change_pct, 2),
                    'volume': int(hist['Volume'].iloc[-1]) if 'Volume' in hist.columns else 0
                }
        except:
            return None
        return None
    
    def quick_scan(self, max_workers=20):
        """Lightning-fast stock scan"""
        print(f"⚡ Quick scanning {len(self.top_stocks)} top BSE stocks...")
        results = []
        
        start_time = time.time()
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_stock = {executor.submit(self.fetch_single_stock_fast, stock): stock for stock in self.top_stocks}
            
            for future in as_completed(future_to_stock):
                result = future.result()
                if result:
                    results.append(result)
        
        end_time = time.time()
        
        if results:
            df = pd.DataFrame(results)
            print(f"✅ Scanned {len(results)} stocks in {end_time - start_time:.1f} seconds")
            return df
        return None
    
    def get_market_snapshot(self):
        """Quick market overview"""
        try:
            sensex = yf.Ticker('^BSESN')
            data = sensex.history(period='2d')
            
            if len(data) >= 1:
                current = data['Close'].iloc[-1]
                prev = data['Close'].iloc[-2] if len(data) > 1 else current
                change = current - prev
                change_pct = (change / prev * 100) if prev != 0 else 0
                
                return {
                    'sensex': round(current, 2),
                    'change': round(change, 2),
                    'change%': round(change_pct, 2)
                }
        except:
            return None
    
    def analyze_quick(self, df):
        """Fast analysis of results"""
        if df is None or df.empty:
            return
        
        print("\n📊 QUICK MARKET ANALYSIS")
        print("=" * 40)
        
        # Market snapshot
        market = self.get_market_snapshot()
        if market:
            print(f"SENSEX: {market['sensex']} ({market['change%']:+.2f}%)")
        
        # Top performers
        print("\n🚀 TOP GAINERS:")
        top_gainers = df.nlargest(5, 'change%')
        for _, row in top_gainers.iterrows():
            print(f"{row['symbol']}: ₹{row['price']} ({row['change%']:+.2f}%)")
        
        print("\n📉 TOP LOSERS:")
        top_losers = df.nsmallest(5, 'change%')
        for _, row in top_losers.iterrows():
            print(f"{row['symbol']}: ₹{row['price']} ({row['change%']:+.2f}%)")
        
        # Quick stats
        positive = len(df[df['change%'] > 0])
        negative = len(df[df['change%'] < 0])
        
        print(f"\n📈 Market Breadth: {positive} up, {negative} down")
        print(f"📊 Average change: {df['change%'].mean():.2f}%")
        
        return df
    
    def plot_quick(self, df):
        """Quick visualization"""
        if df is None or df.empty:
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # Top gainers/losers
        top_5 = df.nlargest(5, 'change%')
        bottom_5 = df.nsmallest(5, 'change%')
        
        ax1.barh(top_5['symbol'], top_5['change%'], color='green', alpha=0.7)
        ax1.set_title('Top 5 Gainers')
        ax1.set_xlabel('Change %')
        
        ax2.barh(bottom_5['symbol'], bottom_5['change%'], color='red', alpha=0.7)
        ax2.set_title('Top 5 Losers')
        ax2.set_xlabel('Change %')
        
        plt.tight_layout()
        plt.show()
    
    def export_quick(self, df):
        """Quick export"""
        if df is None or df.empty:
            return
        
        timestamp = datetime.now().strftime("%H%M%S")
        filename = f"bse_quick_scan_{timestamp}.csv"
        df.to_csv(filename, index=False)
        print(f"\n💾 Data exported to: {filename}")

In [ ]:
# Run ultra-fast BSE analysis
analyzer = FastBSEAnalyzer()

# Quick scan (should complete in 10-15 seconds)
df = analyzer.quick_scan()

# Analyze results
if df is not None:
    analyzer.analyze_quick(df)
    analyzer.plot_quick(df)
    analyzer.export_quick(df)
else:
    print("❌ No data retrieved")

In [ ]:
# Custom quick scan for specific stocks
def quick_custom_scan(symbols):
    """Scan custom list of stocks quickly"""
    print(f"🔍 Custom scanning {len(symbols)} stocks...")
    
    # Add .BO suffix if not present
    formatted_symbols = [s if s.endswith('.BO') else f"{s}.BO" for s in symbols]
    
    analyzer.top_stocks = formatted_symbols
    result_df = analyzer.quick_scan()
    
    if result_df is not None:
        print("\n📋 CUSTOM SCAN RESULTS:")
        print(result_df.sort_values('change%', ascending=False).to_string(index=False))
        return result_df
    return None

# Example: Scan banking stocks
banking_stocks = ['HDFCBANK', 'ICICIBANK', 'SBIN', 'AXISBANK', 'KOTAKBANK']
banking_df = quick_custom_scan(banking_stocks)

In [ ]:
# Quick refresh function
def refresh_data():
    """Refresh market data quickly"""
    print("🔄 Refreshing market data...")
    df = analyzer.quick_scan()
    if df is not None:
        analyzer.analyze_quick(df)
    return df

# Uncomment to refresh
# refresh_data()

In [ ]:
# Performance optimization tips
print("⚡ PERFORMANCE OPTIMIZATIONS APPLIED:")
print("✅ Reduced stock list to top 25 stocks")
print("✅ Minimal data fetching (2 days history only)")
print("✅ Concurrent processing with 20 threads")
print("✅ Simplified data structure")
print("✅ Fast error handling (silent failures)")
print("✅ Optimized pandas settings")
print("\n🚀 Expected execution time: 10-15 seconds")
print("💡 For full BSE 100 analysis, use the previous notebook")
print("💡 This version prioritizes speed over comprehensiveness")